In [1]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split, SBERTPairClassifier_model_selection, Train, TextPairDataset, log_metrics_and_model
import torch
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import os
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
df = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair-class/train-00000-of-00001.parquet")
data = df.to_numpy()
data = data[:5000]

X = data[:, :2]
y = data[:, 2:3]

X_train, X_val, X_test, y_train, y_val, y_test = data_shuffle_split(X, y, 0.15, 0.12, 42)

In [3]:
seed = 42
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size #TODO batch size or number of batches
n_warmup_steps = steps * 0.1
n_freeze = 3

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier_model_selection(model, fc_layer_sizes, device, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [4]:
train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train), batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val), batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test), batch_size = batch_size, shuffle = True, num_workers = 4)

tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
tokenized_train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)

In [ ]:
seed = 42
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size #TODO batch size or number of batches
n_warmup_steps = steps * 0.1
n_freeze = 3

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier_model_selection(model, fc_layer_sizes, device, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
#Fixed sbert model
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = train_loader,
              val_dataloader = validation_loader
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

EPOCH: 0


2025-11-15 11:58:52.173 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 0: train loss = 0.5139845706458784
2025-11-15 11:58:52.174 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 0: validation loss = 0.4046899452805519
2025-11-15 11:58:52.174 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 0: train acc = 0.7141711229946524
2025-11-15 11:58:52.175 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 0: validation acc = 0.792156862745098
2025-11-15 11:58:52.175 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 0: validation f1 = 0.7834967320261438


EPOCH: 1


2025-11-15 11:59:49.553 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 1: train loss = 0.3778985644507612
2025-11-15 11:59:49.554 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 1: validation loss = 0.3939409237354994
2025-11-15 11:59:49.554 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 1: train acc = 0.8213903743315508
2025-11-15 11:59:49.554 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 1: validation acc = 0.7980392156862746
2025-11-15 11:59:49.555 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 1: validation f1 = 0.7864104513720882


EPOCH: 2


...

2025-11-15 12:05:45.587 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 7: train loss = 0.3369646543620998
2025-11-15 12:05:45.589 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 7: validation loss = 0.39655810687690973
2025-11-15 12:05:45.589 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 7: train acc = 0.8481283422459893
2025-11-15 12:05:45.589 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 7: validation acc = 0.7901960784313725
2025-11-15 12:05:45.589 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 7: validation f1 = 0.7797634182073541


EPOCH: 8


2025-11-15 12:06:45.129 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 8: train loss = 0.33717424938311946
2025-11-15 12:06:45.131 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 8: validation loss = 0.3934825547039509
2025-11-15 12:06:45.131 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 8: train acc = 0.8459893048128342
2025-11-15 12:06:45.132 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 8: validation acc = 0.7901960784313725
2025-11-15 12:06:45.132 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 8: validation f1 = 0.7797634182073541


EPOCH: 9


2025-11-15 12:07:46.477 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 9: train loss = 0.336810120787376
2025-11-15 12:07:46.478 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 9: validation loss = 0.39215918630361557
2025-11-15 12:07:46.479 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 9: train acc = 0.8494652406417113
2025-11-15 12:07:46.479 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 9: validation acc = 0.7941176470588235
2025-11-15 12:07:46.480 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 9: validation f1 = 0.7846379061415397


In [ ]:
log_metrics_and_model('SBERTv1', avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, \
                      epoch_val_acc, epoch_val_f1, best_params)

In [5]:
#Trainable Sbert model

seed = 42
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size #TODO batch size or number of batches
n_warmup_steps = steps * 0.1
n_freeze = 3

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier_model_selection(model, fc_layer_sizes, device, dropout_p,trainable=True)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
#Trainable Sbert model

train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = tokenized_train_loader,
              val_dataloader = tokenized_validation_loader,
              sbert_trainable=True
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

EPOCH: 0
